In [1]:
#检索steam新游戏，并且高亮创新性游戏和好评游戏
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.styles import Font, PatternFill


# 定义初始变量
base_url = "https://store.steampowered.com/search/?sort_by=Released_DESC&ignore_preferences=1&supportedlang=schinese&category1=998%2C10&os=win&ndl=1"
game_data = []
start = 0  # 从第一个游戏开始抓取

# 设置请求的超时限制
TIMEOUT = 30  # 设置请求超时时间
MAX_RETRIES = 5  # 最大重试次数

# 请求头部，强制设置为简体中文
headers = {
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8"
}

def get_game_details(game_url, session):
    try:
        if not game_url.startswith('https://store.steampowered.com'):
            game_url = 'https://store.steampowered.com' + game_url

        response = session.get(game_url, headers=headers, timeout=TIMEOUT)
        soup = BeautifulSoup(response.text, 'html.parser')

        # 获取游戏的详细信息，如游戏简介
        description_tag = soup.find('div', class_='game_description_snippet')
        if not description_tag:
            description_tag = soup.find('div', class_='game_description')
        if not description_tag:
            description_tag = soup.find('div', class_='desc_lower')
        
        description = description_tag.text.strip() if description_tag else "无描述"

        # 对长文本进行截断，避免文本过长的问题
        if len(description) > 1000:  # 根据实际需要调整长度
            description = description[:1000] + "..."

        # 获取游戏类型
        genres_tag = soup.find('span', {'data-panel': '{"flow-children":"row"}'})  # 查找类型的 span 标签
        genres = ', '.join([genre.text.strip() for genre in genres_tag.find_all('a')]) if genres_tag else "未知"

        # 获取开发商和发行商
        dev_row = soup.find_all('div', class_='dev_row')
        try:
            developer = dev_row[0].find('a').text.strip() if len(dev_row) > 0 else "未知"
        except IndexError:
            developer = "未知"
        
        try:
            publisher = dev_row[1].find('a').text.strip() if len(dev_row) > 1 else "未知"
        except IndexError:
            publisher = "未知"

        # 获取游戏评测信息
        review_tag = soup.find('span', class_='nonresponsive_hidden responsive_reviewdesc')
        if review_tag:
            reviews_text = review_tag.text.strip()
            reviews_count = ''.join(filter(str.isdigit, reviews_text.split('篇')[0]))  # 获取评测数量
            percentage = ''.join(filter(str.isdigit, reviews_text.split('有')[1].split('%')[0]))  # 获取好评百分比

            game_review = f"此游戏的{reviews_count}篇用户评测中有{percentage}%为好评"
            
            # 评分逻辑
            reviews_count = int(reviews_count)
            percentage = int(percentage)
            
            if percentage >= 95 and reviews_count > 500:
                rating = "好评如潮"
            elif percentage >= 80:
                rating = "特别好评"
            elif percentage >= 70:
                rating = "多半好评"
            elif percentage <= 70 and percentage > 40:
                rating = "褒贬不一"
            elif percentage <= 40 and percentage > 30:
                rating = "多半差评"
            elif percentage <= 30:
                rating = "特别差评"
            elif percentage <= 20 and reviews_count > 500:
                rating = "差评如潮"
            else:
                rating = "特别差评"
        else:
            game_review = "无评测"
            rating = "无评分"

        return description, genres, developer, publisher, rating, game_review
    except Exception as e:
        print(f"获取游戏详情失败：{e}")
        return "无描述", "未知", "未知", "未知", "无评分", "无评测"

def get_game_data(session, start_index=0, max_games=300):
    global game_data
    retries = 0  # 重试次数
    game_count = len(game_data)
    
    while game_count < max_games and retries < MAX_RETRIES:
        try:
            # 构建带有分页参数的URL
            url = f"{base_url}&start={start_index}"
            response = session.get(url, headers=headers, timeout=TIMEOUT)
            soup = BeautifulSoup(response.text, 'html.parser')

            # 找到游戏条目
            game_list = soup.find_all('a', class_='search_result_row')

            for game in game_list:
                if game_count >= max_games:
                    break

                title_tag = game.find('span', class_='title')
                release_date_tag = game.find('div', class_='col search_released')
                if not release_date_tag:
                    release_date_tag = game.find('div', class_='search_released')  # 尝试其他类名

                title = title_tag.text if title_tag else "Unknown"
                release_date = release_date_tag.text.strip() if release_date_tag else "Unknown"

                # 获取每个游戏的详细页面链接
                game_url = game['href']

                # 获取每个游戏的详细信息
                description, genres, developer, publisher, rating, game_review = get_game_details(game_url, session)

                game_data.append({
                    '游戏名称': title,
                    '发行日期': release_date,
                    '游戏描述': description,
                    '用户评价': rating,
                    '游戏评测': game_review,
                    '游戏类型': genres,
                    '开发商': developer,
                    '发行商': publisher,
                    '游戏链接': game_url  # 新增游戏链接字段
                })
                # 打印到控制台
                print(f"游戏名称: {title}, 发行日期: {release_date}, 游戏描述: {description}, 游戏类型: {genres}, 开发商: {developer}, 发行商: {publisher}, 游戏链接: {game_url}, 评分: {rating}, 游戏评测: {game_review}")

                game_count += 1  # 更新游戏数量

            # 如果达到最大游戏数，跳出
            if game_count >= max_games:
                break

            # 增加start参数以模拟翻页
            start_index += len(game_list)

        except Timeout:
            retries += 1
            print(f"请求超时，正在重试 ({retries}/{MAX_RETRIES})...")
            time.sleep(2)  # 等待 2 秒再重试
        except Exception as e:
            print(f"请求失败：{e}")
            break

    return None

# 保存数据到 Excel 文件
# 定义创新性关键词
innovative_keywords = [
    "创新", "新颖", "独特", "前所未有", "突破", "开创", "非凡", "AI",
    "创意", "革新", "前卫", "独一无二", "全新", "非传统", "独特的玩法",
]

def save_data_to_excel():
    # 将爬取到的游戏数据保存到 pandas DataFrame
    df = pd.DataFrame(game_data)

    # 处理发行日期中的空格
    df['发行日期'] = df['发行日期'].str.replace(" ", "")

    # 添加“关注游戏”列，条件为：
    # 1. 游戏描述中包含“创新”，“新颖”，“独特”中的任意一个关键词
    # 2. 或者用户评价列为“好评如潮”，“特别好评”，“多半好评”
    df['关注游戏'] = df.apply(lambda row: '是' if any(keyword in row['游戏描述'] for keyword in innovative_keywords) or row['用户评价'] in ['好评如潮', '特别好评', '多半好评'] else '否', axis=1)

    # 保存到Excel文件
    save_path = r'D:\腾讯\0218steam爬虫\steam_games_detailed.xlsx'
    print(f"正在保存到: {save_path}")
    
    # 保存到Excel文件
    df.to_excel(save_path, index=False)

    # 打开保存的Excel文件，设置字体为华文楷体
    wb = load_workbook(save_path)
    ws = wb.active

    # 设置字体为华文楷体
    font = Font(name="华文楷体", size=12)

    # 高亮显示“关注游戏”的行（背景色为黄色）
    highlight_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")

    for row in ws.iter_rows(min_row=2):  # 跳过标题行
        # 获取“关注游戏”列的值
        follow_game = row[9].value  # 假设“关注游戏”列为第10列

        if follow_game == '是':  # 如果需要关注
            # 高亮显示该行背景
            for cell in row:
                cell.fill = highlight_fill  # 改变背景色为黄色

        # 设置所有单元格字体为华文楷体
        for cell in row:
            cell.font = font

    wb.save(save_path)
    print("数据已保存到 Excel 文件，华文楷体，关注的游戏已标红")

if __name__ == '__main__':
    session = requests.Session()  # 使用 session 来保持连接池
    game_data = []  # 清空旧的数据
    get_game_data(session, start_index=0, max_games=100)  # 爬取300个游戏
    save_data_to_excel()  # 保存数据到 Excel 文件


游戏名称: 逆袭！爆爽打工人！ Demo, 发行日期: 2025 年 2 月 19 日, 游戏描述: 《逆袭！爆爽打工人》是 爽文+网剧题材的文字冒险游戏。暴揍！用巴掌让那些令人不爽的人彻底臣服！, 游戏类型: 动作, 休闲, 独立, 角色扮演, 开发商: 绝不发癫工作室, 发行商: 绝不发癫工作室, 游戏链接: https://store.steampowered.com/app/3384690/_/?snr=1_7_7_230_150_1, 评分: 无评分, 游戏评测: 无评测
游戏名称: SAVE Demo, 发行日期: 2025 年 2 月 19 日, 游戏描述: SAVE是一款生存恐怖游戏。 玩家扮演由米斯卡顿大学签约的PMC特工，开始了一项噩梦般的使命，揭开6000吨级船只"SENARA"的隐藏秘密。, 游戏类型: 动作, 冒险, 独立, 策略, 开发商: Influsion Inc., 发行商: Influsion Inc., 游戏链接: https://store.steampowered.com/app/3227160/SAVE_Demo/?snr=1_7_7_230_150_1, 评分: 无评分, 游戏评测: 无评测
游戏名称: 银河先锋 Demo, 发行日期: 2025 年 2 月 19 日, 游戏描述: 《银河先锋》是一款复古风格的快节奏动作射击游戏，拿起你的武器，消灭无情的敌军与强大的外星怪物。在太空基地、沙漠荒原、熔岩山谷、以及失落的外星遗迹中展开激烈战斗！, 游戏类型: 休闲, 独立, 开发商: TopGame, 发行商: TopGame, 游戏链接: https://store.steampowered.com/app/3308800/_/?snr=1_7_7_230_150_1, 评分: 无评分, 游戏评测: 无评测
游戏名称: Piczle Cross: 符文工房, 发行日期: 2025 年 2 月 19 日, 游戏描述: 通过独特方式在《符文工厂》系列的世界畅游，体验其中的角色和魔物，令人上瘾的数织逻辑谜题一玩就停不下来！, 游戏类型: 休闲, 独立, 开发商: Score Studios LLC, 发行商: Rainy Frog Co. Ltd., 游戏链接: https://store.steampowered.com/ap

In [17]:
#检索steam模拟经营类游戏 并且按照好评排序
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.styles import Font, PatternFill


# 定义初始变量
base_url = "https://store.steampowered.com/search/?sort_by=Reviews_DESC&ignore_preferences=1&supportedlang=schinese&tags=599&category1=998%2C10&ndl=1"
game_data = []
start = 0  # 从第一个游戏开始抓取

# 设置请求的超时限制
TIMEOUT = 30  # 设置请求超时时间
MAX_RETRIES = 5  # 最大重试次数

# 请求头部，强制设置为简体中文
headers = {
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8"
}

def get_game_details(game_url, session):
    try:
        if not game_url.startswith('https://store.steampowered.com'):
            game_url = 'https://store.steampowered.com' + game_url

        response = session.get(game_url, headers=headers, timeout=TIMEOUT)
        soup = BeautifulSoup(response.text, 'html.parser')

        # 获取游戏描述
        description_tag = soup.find('div', class_='game_description_snippet')
        if not description_tag:
            description_tag = soup.find('div', class_='game_description')
        if not description_tag:
            description_tag = soup.find('div', class_='desc_lower')

        description = description_tag.text.strip() if description_tag else "无描述"

        # 扩展的关键词列表，包括“经营”、“建设”、“建造”等
        keywords = [
            "经营", "建设", "建造", "制造", "管理", "顾客", "交易", "商品", "贸易", 
            "店", "商铺", "农场", "工厂", "城市", "餐厅", "动物园", "公园", "医院"
        ]

        # 如果描述中没有任何一个关键词，则跳过该游戏
        if not any(keyword in description for keyword in keywords):
            return None

        # 对长文本进行截断，避免文本过长的问题
        if len(description) > 1000:  # 根据实际需要调整长度
            description = description[:1000] + "..."

        # 获取游戏类型
        genres_tag = soup.find('span', {'data-panel': '{"flow-children":"row"}'})  # 查找类型的 span 标签
        genres = ', '.join([genre.text.strip() for genre in genres_tag.find_all('a')]) if genres_tag else "未知"

        # 获取开发商和发行商
        dev_row = soup.find_all('div', class_='dev_row')
        try:
            developer = dev_row[0].find('a').text.strip() if len(dev_row) > 0 else "未知"
        except IndexError:
            developer = "未知"
        
        try:
            publisher = dev_row[1].find('a').text.strip() if len(dev_row) > 1 else "未知"
        except IndexError:
            publisher = "未知"

        # 获取游戏评测信息
        review_tag = soup.find('span', class_='nonresponsive_hidden responsive_reviewdesc')
        if review_tag:
            reviews_text = review_tag.text.strip()
            reviews_count = ''.join(filter(str.isdigit, reviews_text.split('篇')[0]))  # 获取评测数量
            percentage = ''.join(filter(str.isdigit, reviews_text.split('有')[1].split('%')[0]))  # 获取好评百分比

            game_review = f"此游戏的{reviews_count}篇用户评测中有{percentage}%为好评"
            
            # 评分逻辑
            reviews_count = int(reviews_count)
            percentage = int(percentage)
            
            if percentage >= 95 and reviews_count > 500:
                rating = "好评如潮"
            elif percentage >= 80:
                rating = "特别好评"
            elif percentage >= 70:
                rating = "多半好评"
            elif percentage <= 70 and percentage > 40:
                rating = "褒贬不一"
            elif percentage <= 40 and percentage > 30:
                rating = "多半差评"
            elif percentage <= 30:
                rating = "特别差评"
            elif percentage <= 20 and reviews_count > 500:
                rating = "差评如潮"
            else:
                rating = "特别差评"
        else:
            game_review = "无评测"
            rating = "无评分"

        return description, genres, developer, publisher, rating, game_review
    except Exception as e:
        print(f"获取游戏详情失败：{e}")
        return None  # 出现异常时返回None，表示跳过该游戏




def get_game_data(session, start_index=0, max_games=300):
    global game_data
    retries = 0  # 重试次数
    game_count = len(game_data)
    
    while game_count < max_games and retries < MAX_RETRIES:
        try:
            # 构建带有分页参数的URL
            url = f"{base_url}&start={start_index}"
            response = session.get(url, headers=headers, timeout=TIMEOUT)
            soup = BeautifulSoup(response.text, 'html.parser')

            # 找到游戏条目
            game_list = soup.find_all('a', class_='search_result_row')

            for game in game_list:
                if game_count >= max_games:
                    break

                title_tag = game.find('span', class_='title')
                release_date_tag = game.find('div', class_='col search_released')
                if not release_date_tag:
                    release_date_tag = game.find('div', class_='search_released')  # 尝试其他类名

                title = title_tag.text if title_tag else "Unknown"
                release_date = release_date_tag.text.strip() if release_date_tag else "Unknown"

                # 获取每个游戏的详细页面链接
                game_url = game['href']

                # 获取每个游戏的详细信息
                result = get_game_details(game_url, session)
                
                if result is None:  # 如果返回None，表示该游戏不需要爬取
                    continue

                description, genres, developer, publisher, rating, game_review = result

                game_data.append({
                    '游戏名称': title,
                    '发行日期': release_date,
                    '游戏描述': description,
                    '用户评价': rating,
                    '游戏评测': game_review,
                    '游戏类型': genres,
                    '开发商': developer,
                    '发行商': publisher,
                    '游戏链接': game_url  # 新增游戏链接字段
                })
                # 打印到控制台
                print(f"游戏名称: {title}, 发行日期: {release_date}, 游戏描述: {description}, 游戏类型: {genres}, 开发商: {developer}, 发行商: {publisher}, 游戏链接: {game_url}, 评分: {rating}, 游戏评测: {game_review}")

                game_count += 1  # 更新游戏数量

            # 如果达到最大游戏数，跳出
            if game_count >= max_games:
                break

            # 增加start参数以模拟翻页
            start_index += len(game_list)

        except Timeout:
            retries += 1
            print(f"请求超时，正在重试 ({retries}/{MAX_RETRIES})...")
            time.sleep(2)  # 等待 2 秒再重试
        except Exception as e:
            print(f"请求失败：{e}")
            break

    return None


# 保存数据到 Excel 文件
# 定义创新性关键词
innovative_keywords = [
    "创新", "新颖", "独特", "前所未有", "突破", "开创", "非凡", "AI",
    "创意", "革新", "前卫", "独一无二", "全新", "非传统", "独特的玩法",
]

def save_data_to_excel():
    # 将爬取到的游戏数据保存到 pandas DataFrame
    df = pd.DataFrame(game_data)

    # 处理发行日期中的空格
    df['发行日期'] = df['发行日期'].str.replace(" ", "")

    # 添加“关注游戏”列，条件为：
    # 1. 游戏描述中包含“创新”，“新颖”，“独特”中的任意一个关键词
    # 2. 或者用户评价列为“好评如潮”，“特别好评”，“多半好评”
    df['关注游戏'] = df.apply(lambda row: '是' if any(keyword in row['游戏描述'] for keyword in innovative_keywords) or row['用户评价'] in ['好评如潮', '特别好评', '多半好评'] else '否', axis=1)

    # 保存到Excel文件
    save_path = r'D:\腾讯\0218steam爬虫\steam_games_detailed.xlsx'
    print(f"正在保存到: {save_path}")
    
    # 保存到Excel文件
    df.to_excel(save_path, index=False)

    # 打开保存的Excel文件，设置字体为华文楷体
    wb = load_workbook(save_path)
    ws = wb.active

    # 设置字体为华文楷体
    font = Font(name="华文楷体", size=12)

    # 高亮显示“关注游戏”的行（背景色为黄色）
    highlight_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")

    for row in ws.iter_rows(min_row=2):  # 跳过标题行
        # 获取“关注游戏”列的值
        follow_game = row[9].value  # 假设“关注游戏”列为第10列

        if follow_game == '是':  # 如果需要关注
            # 高亮显示该行背景
            for cell in row:
                cell.fill = highlight_fill  # 改变背景色为黄色

        # 设置所有单元格字体为华文楷体
        for cell in row:
            cell.font = font

    wb.save(save_path)
    print("数据已保存到 Excel 文件，华文楷体，关注的游戏已标红")

if __name__ == '__main__':
    session = requests.Session()  # 使用 session 来保持连接池
    game_data = []  # 清空旧的数据
    get_game_data(session, start_index=0, max_games=100)  # 爬取300个游戏
    save_data_to_excel()  # 保存数据到 Excel 文件


游戏名称: Stardew Valley, 发行日期: 2016 年 2 月 26 日, 游戏描述: 你继承了你爷爷在星露谷留下的老旧农场。带着爷爷留下的残旧工具和几枚硬币开始了你的新生活。你能适应这小镇上的生活并且将这个杂草丛生的老旧农场变成一个繁荣的家吗？, 游戏类型: 独立, 角色扮演, 模拟, 开发商: ConcernedApe, 发行商: ConcernedApe, 游戏链接: https://store.steampowered.com/app/413150/Stardew_Valley/?snr=1_7_7_240_150_1, 评分: 好评如潮, 游戏评测: 此游戏的309231篇用户评测中有98%为好评
游戏名称: 异形工厂2, 发行日期: 2024 年 8 月 15 日, 游戏描述: 深入体验字面意思所示的超大工厂建造世界！建设规模庞大的多层工厂，铺设稳健高效的自动产线，无限制地发挥您的规划才智。按自己的步调应对不断升级的自动化挑战——放轻松点，没有敌人出来捣乱。, 游戏类型: 休闲, 独立, 模拟, 策略, 抢先体验, 开发商: tobspr Games, 发行商: tobspr Games, 游戏链接: https://store.steampowered.com/app/2162800/2/?snr=1_7_7_240_150_1, 评分: 好评如潮, 游戏评测: 此游戏的30167篇用户评测中有96%为好评
游戏名称: Euro Truck Simulator 2, 发行日期: 2012 年 10 月 12 日, 游戏描述: 像公路之王一样在欧洲穿行， 将价值不菲的货物完美送抵远方！往返于英国，比利时，德国，意大利，荷兰，波兰等众多城市；您的耐力，技巧，速度都被您发挥到了极致！, 游戏类型: 独立, 模拟, 开发商: SCS Software, 发行商: SCS Software, 游戏链接: https://store.steampowered.com/app/227300/Euro_Truck_Simulator_2/?snr=1_7_7_240_150_1, 评分: 好评如潮, 游戏评测: 此游戏的307649篇用户评测中有96%为好评
游戏名称: 幸福工厂, 发行日期: 2024 年 9 月 10 日, 游戏描述: 是一款第一人称

In [19]:
!pip install transformers torch


Defaulting to user installation because normal site-packages is not writeable


In [1]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from transformers import pipeline

# 使用Hugging Face的预训练模型进行零样本分类
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# 定义初始变量
base_url = "https://store.steampowered.com/search/?sort_by=Reviews_DESC&ignore_preferences=1&supportedlang=schinese&tags=599&category1=998%2C10&ndl=1"
game_data = []
start = 0  # 从第一个游戏开始抓取

# 设置请求的超时限制
TIMEOUT = 30  # 设置请求超时时间
MAX_RETRIES = 5  # 最大重试次数

# 请求头部，强制设置为简体中文
headers = {
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8"
}

def get_game_details(game_url, session):
    try:
        if not game_url.startswith('https://store.steampowered.com'):
            game_url = 'https://store.steampowered.com' + game_url

        response = session.get(game_url, headers=headers, timeout=TIMEOUT)
        soup = BeautifulSoup(response.text, 'html.parser')

        # 获取游戏描述
        description_tag = soup.find('div', class_='game_description_snippet')
        if not description_tag:
            description_tag = soup.find('div', class_='game_description')
        if not description_tag:
            description_tag = soup.find('div', class_='desc_lower')

        description = description_tag.text.strip() if description_tag else "无描述"

        # 使用AI模型判断游戏是否为模拟经营类
        candidate_labels = ["模拟经营", "动作", "冒险", "角色扮演", "战略", "运动"]
        classification = classifier(description, candidate_labels)

        # 如果分类结果包含“模拟经营”，则继续抓取游戏的详细信息
        if "模拟经营" in classification['labels'][:1]:
            # 获取游戏类型
            genres_tag = soup.find('span', {'data-panel': '{"flow-children":"row"}'})  # 查找类型的 span 标签
            genres = ', '.join([genre.text.strip() for genre in genres_tag.find_all('a')]) if genres_tag else "未知"

            # 获取开发商和发行商
            dev_row = soup.find_all('div', class_='dev_row')
            try:
                developer = dev_row[0].find('a').text.strip() if len(dev_row) > 0 else "未知"
            except IndexError:
                developer = "未知"
            
            try:
                publisher = dev_row[1].find('a').text.strip() if len(dev_row) > 1 else "未知"
            except IndexError:
                publisher = "未知"

            # 获取游戏评测信息
            review_tag = soup.find('span', class_='nonresponsive_hidden responsive_reviewdesc')
            if review_tag:
                reviews_text = review_tag.text.strip()
                reviews_count = ''.join(filter(str.isdigit, reviews_text.split('篇')[0]))  # 获取评测数量
                percentage = ''.join(filter(str.isdigit, reviews_text.split('有')[1].split('%')[0]))  # 获取好评百分比

                game_review = f"此游戏的{reviews_count}篇用户评测中有{percentage}%为好评"
                
                # 评分逻辑
                reviews_count = int(reviews_count)
                percentage = int(percentage)
                
                if percentage >= 95 and reviews_count > 500:
                    rating = "好评如潮"
                elif percentage >= 80:
                    rating = "特别好评"
                elif percentage >= 70:
                    rating = "多半好评"
                elif percentage <= 70 and percentage > 40:
                    rating = "褒贬不一"
                elif percentage <= 40 and percentage > 30:
                    rating = "多半差评"
                elif percentage <= 30:
                    rating = "特别差评"
                elif percentage <= 20 and reviews_count > 500:
                    rating = "差评如潮"
                else:
                    rating = "特别差评"
            else:
                game_review = "无评测"
                rating = "无评分"

            return description, genres, developer, publisher, rating, game_review
        else:
            return None  # 如果不是模拟经营类游戏，则跳过该游戏

    except Exception as e:
        print(f"获取游戏详情失败：{e}")
        return None  # 出现异常时返回None，表示跳过该游戏


def get_game_data(session, start_index=0, max_games=300):
    global game_data
    retries = 0  # 重试次数
    game_count = len(game_data)
    
    while game_count < max_games and retries < MAX_RETRIES:
        try:
            # 构建带有分页参数的URL
            url = f"{base_url}&start={start_index}"
            response = session.get(url, headers=headers, timeout=TIMEOUT)
            soup = BeautifulSoup(response.text, 'html.parser')

            # 找到游戏条目
            game_list = soup.find_all('a', class_='search_result_row')

            for game in game_list:
                if game_count >= max_games:
                    break

                title_tag = game.find('span', class_='title')
                release_date_tag = game.find('div', class_='col search_released')
                if not release_date_tag:
                    release_date_tag = game.find('div', class_='search_released')  # 尝试其他类名

                title = title_tag.text if title_tag else "Unknown"
                release_date = release_date_tag.text.strip() if release_date_tag else "Unknown"

                # 获取每个游戏的详细页面链接
                game_url = game['href']

                # 获取每个游戏的详细信息
                result = get_game_details(game_url, session)
                
                if result is None:  # 如果返回None，表示该游戏不需要爬取
                    continue

                description, genres, developer, publisher, rating, game_review = result

                game_data.append({
                    '游戏名称': title,
                    '发行日期': release_date,
                    '游戏描述': description,
                    '用户评价': rating,
                    '游戏评测': game_review,
                    '游戏类型': genres,
                    '开发商': developer,
                    '发行商': publisher,
                    '游戏链接': game_url  # 新增游戏链接字段
                })
                # 打印到控制台
                print(f"游戏名称: {title}, 发行日期: {release_date}, 游戏描述: {description}, 游戏类型: {genres}, 开发商: {developer}, 发行商: {publisher}, 游戏链接: {game_url}, 评分: {rating}, 游戏评测: {game_review}")

                game_count += 1  # 更新游戏数量

            # 如果达到最大游戏数，跳出
            if game_count >= max_games:
                break

            # 增加start参数以模拟翻页
            start_index += len(game_list)

        except Timeout:
            retries += 1
            print(f"请求超时，正在重试 ({retries}/{MAX_RETRIES})...")
            time.sleep(2)  # 等待 2 秒再重试
        except Exception as e:
            print(f"请求失败：{e}")
            break

    return None


# 保存数据到 Excel 文件
def save_data_to_excel():
    df = pd.DataFrame(game_data)
    
    # 按照好评排序
    df_sorted = df.sort_values(by='用户评价', ascending=False)

    # 保存到Excel文件
    save_path = r'D:\腾讯\0218steam爬虫\steam_games_detailed.xlsx'
    print(f"正在保存到: {save_path}")
    
    # 保存到Excel文件
    df_sorted.to_excel(save_path, index=False)

    # 打开保存的Excel文件，设置字体为华文楷体
    wb = load_workbook(save_path)
    ws = wb.active
    font = Font(name="华文楷体", size=12)

    # 高亮显示“关注游戏”的行（背景色为黄色）
    highlight_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")

    for row in ws.iter_rows(min_row=2):  # 跳过标题行
        # 获取“关注游戏”列的值
        follow_game = row[9].value  # 假设“关注游戏”列为第10列

        if follow_game == '是':  # 如果需要关注
            # 高亮显示该行背景
            for cell in row:
                cell.fill = highlight_fill  # 改变背景色为黄色

        # 设置所有单元格字体为华文楷体
        for cell in row:
            cell.font = font

    wb.save(save_path)
    print("数据已保存到 Excel 文件，华文楷体，关注的游戏已标红")

if __name__ == '__main__':
    session = requests.Session()  # 使用 session 来保持连接池
    game_data = []  # 清空旧的数据
    get_game_data(session, start_index=0, max_games=100)  # 爬取300个游戏
    save_data_to_excel()  # 保存数据到 Excel 文件


KeyboardInterrupt: 

In [41]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\goodmanliu\\.cache\\huggingface\\hub\\models--facebook--bart-large-mnli\\snapshots\\d7645e127eaf1aefc7862fd59a17a5aa8558b8ce\\config.json'

In [3]:
from transformers import pipeline

# 指定本地模型文件夹路径
local_model_path = "C:\\Users\\goodmanliu\\.cache\\huggingface\\hub\\mnli bart ceshi"  # 修改为你本地的路径

# 加载本地模型
classifier = pipeline("zero-shot-classification", model=local_model_path)

# 示例文本和候选标签
text = "在一个农场中经营农场，种植和销售农作物。"
candidate_labels = ["模拟经营", "冒险", "角色扮演", "战略"]

# 进行分类
result = classifier(text, candidate_labels)

# 输出结果
print(result)


Device set to use cpu


{'sequence': '在一个农场中经营农场，种植和销售农作物。', 'labels': ['模拟经营', '战略', '角色扮演', '冒险'], 'scores': [0.6282685995101929, 0.16245201230049133, 0.14052723348140717, 0.06875216215848923]}


In [19]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# 本地路径加载模型和分词器
model_path = "C:\\Users\\goodmanliu\\.cache\\huggingface\\hub\\MoritzLaurerDeBERTa-v3-base-mnli-fever-anli"

# 加载模型和分词器
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# 使用pipeline进行零样本分类
classifier = pipeline("zero-shot-classification", model=model, tokenizer=tokenizer)

# 设置请求的超时限制
TIMEOUT = 30  # 设置请求超时时间
MAX_RETRIES = 5  # 最大重试次数

# 请求头部，强制设置为简体中文
headers = {
    "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

# 定义初始变量
base_url = "https://store.steampowered.com/search/?sort_by=Reviews_DESC&ignore_preferences=1&supportedlang=schinese&tags=599&category1=998%2C10&ndl=1"
game_data = []

def get_game_details(game_url, session):
    try:
        response = session.get(game_url, headers=headers, timeout=TIMEOUT)
        soup = BeautifulSoup(response.text, 'html.parser')

        # 获取游戏描述
        description_tag = soup.find('div', class_='game_description_snippet')
        description = description_tag.text.strip() if description_tag else "无描述"

        # 使用AI模型判断游戏是否为模拟经营类
        candidate_labels = ["模拟经营", "动作", "冒险", "射击", "格斗", "模拟器", "角色扮演", "载具驾驶", "策略", "音乐", "赛车", "休闲", "非游戏向", "其他"]
        classification = classifier(description, candidate_labels)

        # 输出AI分类的标签和权重
        print(f"游戏描述: {description}")
        print(f"分类结果: {classification}")

        game_type = classification['labels'][0]
        weight = classification['scores'][0]

        # 获取游戏类型
        genres_tag = soup.find('span', {'data-panel': '{"flow-children":"row"}'})  # 查找类型的 span 标签
        genres = ', '.join([genre.text.strip() for genre in genres_tag.find_all('a')]) if genres_tag else "未知"

        # 获取开发商和发行商
        dev_row = soup.find_all('div', class_='dev_row')
        developer = dev_row[0].find('a').text.strip() if len(dev_row) > 0 else "未知"
        publisher = dev_row[1].find('a').text.strip() if len(dev_row) > 1 else "未知"

        # 获取游戏评测信息
        review_tag = soup.find('span', class_='nonresponsive_hidden responsive_reviewdesc')
        if review_tag:
            reviews_text = review_tag.text.strip()
            reviews_count = ''.join(filter(str.isdigit, reviews_text.split('篇')[0]))  # 获取评测数量
            percentage = ''.join(filter(str.isdigit, reviews_text.split('有')[1].split('%')[0]))  # 获取好评百分比

            game_review = f"此游戏的{reviews_count}篇用户评测中有{percentage}%为好评"
            
            # 评分逻辑
            reviews_count = int(reviews_count)
            percentage = int(percentage)
            
            if percentage >= 95 and reviews_count > 500:
                rating = "好评如潮"
            elif percentage >= 80:
                rating = "特别好评"
            elif percentage >= 70:
                rating = "多半好评"
            elif percentage <= 70 and percentage > 40:
                rating = "褒贬不一"
            elif percentage <= 40 and percentage > 30:
                rating = "多半差评"
            elif percentage <= 30:
                rating = "特别差评"
            elif percentage <= 20 and reviews_count > 500:
                rating = "差评如潮"
            else:
                rating = "特别差评"
        else:
            game_review = "无评测"
            rating = "无评分"

        return description, genres, developer, publisher, rating, game_review, game_type, weight
    except Exception as e:
        print(f"获取游戏详情失败：{e}")
        return None

def get_game_data(session, start_index=0, max_games=20):
    global game_data
    retries = 0
    game_count = len(game_data)
    
    while game_count < max_games and retries < MAX_RETRIES:
        try:
            # 构建带有分页参数的URL
            url = f"{base_url}&start={start_index}"
            response = session.get(url, headers=headers, timeout=TIMEOUT)
            soup = BeautifulSoup(response.text, 'html.parser')

            # 找到游戏条目
            game_list = soup.find_all('a', class_='search_result_row')

            for game in game_list:
                if game_count >= max_games:
                    break

                title_tag = game.find('span', class_='title')
                release_date_tag = game.find('div', class_='col search_released')
                title = title_tag.text if title_tag else "Unknown"
                release_date = release_date_tag.text.strip() if release_date_tag else "Unknown"

                # 获取每个游戏的详细页面链接
                game_url = game['href']

                # 获取每个游戏的详细信息
                result = get_game_details(game_url, session)
                
                if result is None:
                    continue

                description, genres, developer, publisher, rating, game_review, game_type, weight = result

                game_data.append({
                    '游戏名称': title,
                    '发行日期': release_date,
                    '游戏描述': description,
                    '用户评价': rating,
                    '游戏评测': game_review,
                    '游戏类型': genres,
                    '开发商': developer,
                    '发行商': publisher,
                    'AI分类': game_type,
                    '分类权重': weight,
                    '游戏链接': game_url
                })
                print(f"游戏名称: {title}, 发行日期: {release_date}, 游戏描述: {description}, 游戏类型: {genres}, 开发商: {developer}, 发行商: {publisher}, 游戏链接: {game_url}, 评分: {rating}, 游戏评测: {game_review}, AI分类: {game_type}, 分类权重: {weight}")

                game_count += 1

            if game_count >= max_games:
                break

            start_index += len(game_list)

        except Timeout:
            retries += 1
            print(f"请求超时，正在重试 ({retries}/{MAX_RETRIES})...")
            time.sleep(2)
        except Exception as e:
            print(f"请求失败：{e}")
            break

    return None

# 保存数据到 Excel 文件
def save_data_to_excel():
    df = pd.DataFrame(game_data)
    
    # 按照好评排序
    df_sorted = df.sort_values(by='用户评价', ascending=False)

    # 保存到Excel文件
    save_path = r'D:\腾讯\0218steam爬虫\steam_games_detailed.xlsx'
    print(f"正在保存到: {save_path}")
    
    df_sorted.to_excel(save_path, index=False)

    wb = load_workbook(save_path)
    ws = wb.active
    font = Font(name="华文楷体", size=12)

    highlight_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")

    for row in ws.iter_rows(min_row=2):  # 跳过标题行
        follow_game = row[9].value  # 假设“关注游戏”列为第10列

        if follow_game == '是':
            for cell in row:
                cell.fill = highlight_fill

        for cell in row:
            cell.font = font

    wb.save(save_path)
    print("已保存，华文楷体，标黄")

if __name__ == '__main__':
    session = requests.Session()
    game_data = []
    get_game_data(session, start_index=0, max_games=20)
    save_data_to_excel()


Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


游戏描述: 你继承了你爷爷在星露谷留下的老旧农场。带着爷爷留下的残旧工具和几枚硬币开始了你的新生活。你能适应这小镇上的生活并且将这个杂草丛生的老旧农场变成一个繁荣的家吗？
分类结果: {'sequence': '你继承了你爷爷在星露谷留下的老旧农场。带着爷爷留下的残旧工具和几枚硬币开始了你的新生活。你能适应这小镇上的生活并且将这个杂草丛生的老旧农场变成一个繁荣的家吗？', 'labels': ['非游戏向', '动作', '角色扮演', '策略', '模拟经营', '载具驾驶', '模拟器', '冒险', '音乐', '格斗', '其他', '休闲', '射击', '赛车'], 'scores': [0.5103151202201843, 0.16558915376663208, 0.09729740023612976, 0.05933762714266777, 0.04281711205840111, 0.041722774505615234, 0.02844950743019581, 0.027884509414434433, 0.006314133759588003, 0.0055787996388971806, 0.005469480995088816, 0.003873674664646387, 0.0030519147403538227, 0.0022988179698586464]}
游戏名称: Stardew Valley, 发行日期: Unknown, 游戏描述: 你继承了你爷爷在星露谷留下的老旧农场。带着爷爷留下的残旧工具和几枚硬币开始了你的新生活。你能适应这小镇上的生活并且将这个杂草丛生的老旧农场变成一个繁荣的家吗？, 游戏类型: 独立, 角色扮演, 模拟, 开发商: ConcernedApe, 发行商: ConcernedApe, 游戏链接: https://store.steampowered.com/app/413150/Stardew_Valley/?snr=1_7_7_240_150_1, 评分: 好评如潮, 游戏评测: 此游戏的309179篇用户评测中有98%为好评, AI分类: 非游戏向, 分类权重: 0.5103151202201843
游戏描述: Balatro是一场循环不息的扑克之旅。在这款令人陶醉的策略构建游戏中，你将参与

In [15]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("zero-shot-classification", model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\goodmanliu\\.cache\\huggingface\\hub\\models--MoritzLaurer--DeBERTa-v3-base-mnli-fever-anli\\snapshots\\6f5cf0a2b59cabb106aca4c287eed12e357e90eb\\config.json'